In [2]:
%%capture --no-stderr
%pip install --upgrade --force-reinstall langgraph
%pip install --quiet -U langchain_openai langchain_core
%pip install requests

In [3]:
pip install "langsmith[otel]"

In [4]:
import os
os.environ["LANGSMITH_OTEL_ENABLED"] = "true"
os.environ["LANGSMITH_TRACING"]= "true"
os.environ["LANGSMITH_ENDPOINT"]= "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"]= "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"

curl https://llmexecution.api.intuit.com/v3/models

## Generate an Access Token by running the following curl command either in the terminal or postman/intuit api client

### Note - First replace the app_secret in the command with secret shared in TinCan







```
curl --location 'https://identityinternal.api.intuit.com/signin/graphql' --header 'Authorization: Intuit_IAM_Authentication intuit_appid="Intuit.aifabric.genos.agenticaitrainingclient",intuit_app_secret=prdFtgimomcXJHvKvJo80831nB6FINieM2HKkGwO' --header 'Content-Type: application/json' --data-raw '{"query":"mutation identityTestSignInWithPassword($input: Identity_TestSignInWithPasswordInput!) {\n identityTestSignInWithPassword(input: $input) {\n accessToken\n legacyAuthId\n}\n}","variables":{"input":{"username":"iamtestpass_616829375333307","password":"Intuit01-","tenantId":"50000003","intent":{"appGroup":"QBO","assetAlias":"Intuit.aifabric.genos.agenticaitrainingclient"}}}}'
```




## Generate the PrivateAuth+ headers for calling the LLM
### Note - First replace the intuit_app_secret and intuit_token fields in the AUTHN_STRING below with the app_secret shared in TinCan and accessToken fromt the previous step

In [18]:
# Intuit PrivateAuth+ headers
# Use the Access Token from the previous cell in the intuit_token_type field
AUTHN_STRING = "Intuit_IAM_Authentication " \
               "intuit_appid='Intuit.aifabric.genos.agenticaitrainingclient'," \
               "intuit_app_secret='prdFtgimomcXJHvKvJo80831nB6FINieM2HKkGwO'," \
               "intuit_token_type='IAM-Ticket'," \
               f"intuit_userid=4621097838312242983," \
               f"intuit_token=V1-32-B0iteu1r8llg2nvbmq2f9d"

INTUIT_AUTHN_HEADERS = {
    "Authorization": AUTHN_STRING,
    "intuit_experience_id": "c1392e45-9ef3-4d87-9e6f-3dd45294e545", # GenOS experience_id
    "intuit_originating_assetalias": "Intuit.aifabric.genos.agenticaitrainingclient"
}

## Define tools and bind to LLM

In [19]:
from langchain_openai import ChatOpenAI
def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second intfrom langchain_openai import ChatOpenAI
    """
    return a * b

# This will be a tool
def add(a: int, b: int) -> int:
    """Adds a and b.

    Args:
        a: first int
        b: second int
    """
    return a + b

def divide(a: int, b: int) -> float:
    """Divide a and b.

    Args:
        a: first int
        b: second int
    """
    return a / b

tools = [add, multiply, divide]

llm = ChatOpenAI(
    api_key="anything",  # No OP as we will use Intuit AuthN headers
    model="xxxx",  # No OP as model id is in path param
    base_url="https://llmexecution.api.intuit.com/v3/gpt-4o-2024-05-13/",
    temperature=0,
    max_tokens=4096,
    extra_headers={
        **INTUIT_AUTHN_HEADERS
    })

llm_with_tools = llm.bind_tools(tools)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3473: UserWarning: WARNING! extra_headers is not default parameter.
                extra_headers was transferred to model_kwargs.
                Please confirm that extra_headers is what you intended.
  if (await self.run_code(code, result,  async_=asy)):


## Run LLM using langchain without any tool

In [ ]:
message = [
    ('system', 'please assist user in calculating'),
    ('user', 'what is 3747345/347 Think step by step.')
]

results = llm.invoke(message)
print(results.text)

To calculate \( \frac{3747345}{347} \) step by step, follow these steps:

1. **Estimate the quotient**: Start by estimating how many times 347 can fit into 3747345. This can be done by simplifying the numbers.

2. **Divide the numbers**: Perform the division using long division.

3. **Calculate the quotient**: Find the quotient by dividing the numbers directly.

Let's go through the steps:

1. **Estimate the quotient**:
   - 3747345 is approximately 3.75 million.
   - 347 is approximately 350.
   - Estimate: \( \frac{3,750,000}{350} \approx \frac{3,750,000}{350} = 10,714.29 \).

2. **Divide the numbers**:
   - Perform long division of 3747345 by 347.

3. **Calculate the quotient**:
   - \( 3747345 \div 347 \approx 10800 \).

Let's perform the long division to get the exact quotient:

```
       10800
    ___________
347 | 3747345
      -347
    _______
       2773
      -2082
    _______
        6914
      -6940
    _______
        2975
      -2776
    _______
         1995
      -1735

## Tool calling under the hood

In [ ]:
results = llm.invoke([
    ('system', 'Please assist the user in calculating'),
    ('user', 'What is 234293874 / 342, use a tool to calculate this, do not perform the calculation manually, by returning JSON in the following format: { "function_call": "divide", "numerator": 5, "divisor": 2}')
])

print(results.content)

{
  "function_call": "divide",
  "numerator": 234293874,
  "divisor": 342
}


In [ ]:
results.content.split("```json")

['{\n  "function_call": "divide",\n  "numerator": 234293874,\n  "divisor": 342\n}']

In [ ]:
json_results = results.content.split("```json")[0].split("```")[0]
import json
print(json.loads(json_results))
tool_call = json.loads(json_results)

{'function_call': 'divide', 'numerator': 234293874, 'divisor': 342}


In [ ]:
tool_call

{'function_call': 'divide', 'numerator': 234293874, 'divisor': 342}

In [ ]:
def divide(numerator,divisor):
  return numerator / divisor
tools = {
    "divide": divide
}

result = tools[tool_call["function_call"]](tool_call["numerator"], tool_call["divisor"])
print(result)

685069.8070175438


In [ ]:
results = llm.invoke([
    ('system', 'Please assist the user in calculating'),
    ('user', 'What is 234293874 / 342, use a tool to calculate this, do not perform the calculation manually, by returning JSON in the following format: { "function_call": "divide", "numerator": 5, "divisor": 2}'),
    ('assistant', results.content),
    ('user', "The tool call's result is: " + str(result))
])
print(results)

content='{\n  "result": 685069.8070175438\n}' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 122, 'total_tokens': 138, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'text_tokens': None, 'image_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_ee1d74bde0', 'id': 'chatcmpl-CbCIeYqW3lym5eAphk5H6Y492Pnb1', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--c4b49c8f-bd15-467a-abf5-d1995eff911a-0' usage_metadata={'input_tokens': 122, 'output_tokens': 16, 'total_tokens': 138, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


## Run LLM using langchain with tools

In [7]:
message = [
    ('system', 'please assist user in calculating'),
    ('user', 'what is 3747345/347 Think step by step.')
]

results = llm_with_tools.invoke(message)
print(results)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 138, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'text_tokens': None, 'image_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_ee1d74bde0', 'id': 'chatcmpl-CbWGZQ7hcu5G4B3xGalhgiFetrAiG', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--38ca75d5-6e7f-4dd2-a1ce-0ffdbcf2ad43-0' tool_calls=[{'name': 'divide', 'args': {'a': 3747345, 'b': 347}, 'id': 'call_VFavsiNwZtxvXKofTdvhuvuv', 'type': 'tool_call'}] usage_metadata={'input_tokens': 138, 'output_tokens': 20, 'total_tokens': 158, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


## Add assistant

In [20]:
from langgraph.graph import MessagesState
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# System message
sys_msg = SystemMessage(content="You are a helpful assistant tasked with performing arithmetic on a set of inputs.")

# Node
def assistant(state: MessagesState):
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

## Build the graph

In [ ]:
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition, ToolNode
from IPython.display import Image, display

# Graph
builder = StateGraph(MessagesState)

# Define nodes: these do the work
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# Define edges: these determine how the control flow moves
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition,
)
builder.add_edge("tools", "assistant")

checkpointer = InMemorySaver()


react_graph = builder.compile(debug=True, checkpointer=checkpointer)

# Show
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

## Invoke the agent

In [22]:
## tracer goes here
config: RunnableConfig = {"configurable": {"thread_id": "2"}}
messages = [HumanMessage(content="Add 2232 and 222222, then divide the whole thing by 14.")]
messages = react_graph.invoke({"messages": messages}, config)
for m in messages['messages']:
    m.pretty_print()

[values] {'messages': [HumanMessage(content='Add 389573 and 4439374. Then divide the whole thing by 6', additional_kwargs={}, response_metadata={}, id='f71eb70e-d2ce-4e73-9390-bf6e9428df44')]}
[updates] {'assistant': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 152, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'text_tokens': None, 'image_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_3eed281ddb', 'id': 'chatcmpl-CbXo5DNW8IqdEMzvlJZCKfwDXUQOa', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--69e2a3b8-6d91-45ce-a19a-b8ef11afdb77-0', tool_calls=[{'name': 'add', 'args': {'a': 389573, 'b': 4439374}, 'id': 'call_1g8YqdD

In [24]:
config = {"configurable": {"thread_id": "1"}}
react_graph.get_state(config)

ValueError: No checkpointer set